In [1]:
import numpy as np
import pandas as pd
import h5py
from sklearn.decomposition import IncrementalPCA
import joblib
import os
import scanpy as sc

In [4]:
results_dir = "combined_UCE_20M_Brain_NervousSystem_shuffled"

n_jobs = 8
n_components = 50
chunk_size = 100_000
n_neighbors = 20

uce_outfile = "uce.h5"
#group_name = "Brain_NervousSystem"
group_name = ""
ipca_outfile = f"ipca_model_{group_name}.pkl" if group_name else "ipca_model.pkl"
pca_outfile = f"pca_{group_name}.h5" if group_name else "pca.h5"

In [5]:
# handle to uce.h5 
fuce = h5py.File(os.path.join(results_dir, uce_outfile), "r")
uce = fuce["data"] # h5 handle
print(uce.shape)

(20019666, 1280)


## IncrementalPCA - Fit
assuming all data already preshuffled

In [6]:
# Fit PCA in streaming mode
ipca = IncrementalPCA(n_components=n_components)

iteration =0
total_len = uce.shape[0]
#chunks_iter = int(batch_size/chunk_size)
for i in range(0, total_len, chunk_size):
    iteration = iteration + 1
    print ("processing: cell", i, iteration)
    batch = uce[i:i+chunk_size, :]
    ipca.partial_fit(batch)

processing: cell 0 1
processing: cell 100000 2
processing: cell 200000 3
processing: cell 300000 4
processing: cell 400000 5
processing: cell 500000 6
processing: cell 600000 7
processing: cell 700000 8
processing: cell 800000 9
processing: cell 900000 10
processing: cell 1000000 11
processing: cell 1100000 12
processing: cell 1200000 13
processing: cell 1300000 14
processing: cell 1400000 15
processing: cell 1500000 16
processing: cell 1600000 17
processing: cell 1700000 18
processing: cell 1800000 19
processing: cell 1900000 20
processing: cell 2000000 21
processing: cell 2100000 22
processing: cell 2200000 23
processing: cell 2300000 24
processing: cell 2400000 25
processing: cell 2500000 26
processing: cell 2600000 27
processing: cell 2700000 28
processing: cell 2800000 29
processing: cell 2900000 30
processing: cell 3000000 31
processing: cell 3100000 32
processing: cell 3200000 33
processing: cell 3300000 34
processing: cell 3400000 35
processing: cell 3500000 36
processing: cell

In [7]:
# save PCA fit model -- since it takes a long time to run
joblib.dump(ipca, os.path.join(results_dir, ipca_outfile))

['combined_UCE_20M_Brain_NervousSystem_shuffled/ipca_model.pkl']

## IncrementalPCA - Transform

In [ ]:
# read model back
'''
ipca = joblib.load(os.path.join(results_dir, ipca_outfile))
ipca_outfile
'''

In [8]:
# Apply ipca.transform in batches

# Create output array (e.g., 36M × 50)
#fpca.close()
n_rows = uce.shape[0]
fpca =  h5py.File(os.path.join(results_dir, pca_outfile), "w")
batch_size = min(chunk_size, n_rows)

print (batch_size)
X_pca = fpca.create_dataset(
    "pca",
    shape=(n_rows, ipca.n_components),
    dtype="float32",
    chunks=(batch_size, ipca.n_components),
    compression="gzip"
)

# incremental transform
for i in range(0, n_rows, batch_size):
    print ("Processing batch ", i//batch_size)
    batch = uce[i:i + batch_size,:]
    print (batch.shape)
    transformed = ipca.transform(batch)
    X_pca[i:i + len(transformed)] = transformed
    print(f"Transformed rows {i} to {i + len(transformed)}")
print("PCA output successfully saved to disk.")
print(X_pca.shape)

100000
Processing batch  0
(100000, 1280)
Transformed rows 0 to 100000
Processing batch  1
(100000, 1280)
Transformed rows 100000 to 200000
Processing batch  2
(100000, 1280)
Transformed rows 200000 to 300000
Processing batch  3
(100000, 1280)
Transformed rows 300000 to 400000
Processing batch  4
(100000, 1280)
Transformed rows 400000 to 500000
Processing batch  5
(100000, 1280)
Transformed rows 500000 to 600000
Processing batch  6
(100000, 1280)
Transformed rows 600000 to 700000
Processing batch  7
(100000, 1280)
Transformed rows 700000 to 800000
Processing batch  8
(100000, 1280)
Transformed rows 800000 to 900000
Processing batch  9
(100000, 1280)
Transformed rows 900000 to 1000000
Processing batch  10
(100000, 1280)
Transformed rows 1000000 to 1100000
Processing batch  11
(100000, 1280)
Transformed rows 1100000 to 1200000
Processing batch  12
(100000, 1280)
Transformed rows 1200000 to 1300000
Processing batch  13
(100000, 1280)
Transformed rows 1300000 to 1400000
Processing batch  1

In [9]:
fpca.close()
fuce.close()

## UMAP (non-parametric)

In [ ]:
# load precomputed Knn
data = np.load(os.path.join(results_dir, "knn_30.npz"))
indices = data["indices"]
distances = data["distances"]
precomputed_knn = (indices, distances)

In [ ]:
# Run UMAP using the precomputed neighbors
import umap
reducer = umap.UMAP(
    n_neighbors=n_neighbors,
    n_components=2,
    precomputed_knn=precomputed_knn,
    metric="precomputed",  # important
    n_jobs = n_jobs,
    # random_state=42,
    verbose=True
)

X_umap = reducer.fit_transform(X_pca)

In [ ]:
X_umap.shape

In [ ]:
# Save UMAP
with h5py.File(os.path.join(results_dir, "umap.h5"), "w") as f:
    f.create_dataset("umap", data=X_umap, compression="gzip")

## UMAP viz quick check

In [ ]:
# # read umap back
fumap = h5py.File(os.path.join(results_dir, "umap.h5"), "r")
X_umap = fumap["umap"] # h5 handle
print(X_umap.shape)

In [ ]:
import matplotlib.pyplot as plt

# downsample
#indices = np.random.choice(X_umap.shape[0], size=500_000, replace=False)
plt.scatter(X_umap[:, 0], X_umap[:, 1], s=0.1, alpha=0.5)
plt.axis('off')
plt.title("UMAP projection")
plt.show()

In [ ]:
# get post uce gene expression data
import anndata
adata = anndata.read_h5ad(os.path.join(results_dir, "c4b03352-af8d-492a-8d6b-40f304e0a122_uce_adata.h5ad"))
adata

In [ ]:
# umap using gene exprssion by myslef
import scanpy as sc

# 1. Normalize
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# 2. Highly variable genes
sc.pp.highly_variable_genes(adata)
adata = adata[:, adata.var["highly_variable"]]

# 3. Scale (optional but common)
sc.pp.scale(adata, max_value=10)

# 4. PCA
sc.tl.pca(adata, svd_solver="arpack")

# 5. Compute neighbors
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)

# 6. Run UMAP
sc.tl.umap(adata)

# 7. Plot UMAP
sc.pl.umap(adata, color="DRD1")  # or any gene, cluster, metadata
adata

In [ ]:
adata.obsm['X_uce_umap']=X_umap
adata

In [ ]:
adata.write_h5ad(os.path.join(results_dir, "c4b03352-af8d-492a-8d6b-40f304e0a122_uce_adata.h5ad"))

In [ ]:
adata.var

In [ ]:
import scanpy as sc
sc.pl.embedding(adata, "X_umap", color=["DRD1","DRD2","dissection"], gene_symbols="Gene", vmin=0, vmax=3, legend_loc=None)

In [ ]:
import scanpy as sc
sc.pl.embedding(adata, "X_uce_umap", color=["DRD1","DRD2","dissection"], gene_symbols="Gene", vmin=0, vmax=3, legend_loc=None)

get cell type data

In [ ]:
# cell type
cell_type = pd.read_csv(os.path.join(results_dir, "cell_type.tsv.gz"), sep="\t", compression='gzip')

In [ ]:
cell_type.head()

In [ ]:
assert (len(cell_type) == len(X_umap))

In [ ]:
# Extract the cell_type column
labels = cell_type["cell_type"].astype(str).values[indices,]
labels.shape

In [ ]:
# Encode cell type labels to integers for color mapping
encoded_labels, unique_labels = pd.factorize(labels)
len(unique_labels)

In [ ]:
import matplotlib.patches as mpatches

plt.scatter(X_umap[indices, 0], X_umap[indices, 1], 
            c=encoded_labels, cmap="tab20", 
            s=0.1, alpha=0.5)
plt.axis('off')
plt.title("UMAP colored by cell type (500K subset)")

'''
# Create custom legend handles
legend_elements = [
    mpatches.Patch(color=plt.cm.tab20(i / len(unique_labels)), label=label)
    for i, label in enumerate(unique_labels)
]

# Add legend (adjust number of columns or fontsize as needed)
plt.legend(handles=legend_elements, title="Cell Type", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
plt.tight_layout()
'''

plt.show()